# MOP 3231 · Task 2 — Decision Trees
SEED = 3194

In [2]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.5 2.0.2 2.3.3 1.6.1


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3194

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


In [4]:
import numpy as np


def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    p = counts[counts > 0] / len(y)
    return float(-np.sum(p * np.log2(p)))


def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    y = np.asarray(y)
    left = np.asarray(left, dtype=bool)
    n = len(y)
    n_left = left.sum()
    n_right = n - n_left
    if n_left == 0 or n_right == 0:
        return 0.0
    return entropy(y) - (n_left / n) * entropy(y[left]) - (n_right / n) * entropy(y[~left])


def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    X = np.asarray(X)
    y = np.asarray(y)
    best = None  # (gain, feature, threshold)
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        if len(values) < 2:
            continue
        thresholds = (values[:-1] + values[1:]) / 2.0
        for t in thresholds:
            left = X[:, j] <= t
            n_left = left.sum()
            if n_left < min_samples_leaf or len(y) - n_left < min_samples_leaf:
                continue
            g = information_gain(y, left)
            # strictly greater: ties keep the earlier (lower feature, lower threshold) candidate
            if best is None or g > best[0] + 1e-12:
                best = (g, j, t)
    if best is None:
        return None
    return best[1], float(best[2]), float(best[0])


def _majority(y):
    """Majority class; a tie goes to the smaller label."""
    return int(np.argmax(np.bincount(y)))


def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts (or any structure you like)."""
    X = np.asarray(X)
    y = np.asarray(y)
    if len(np.unique(y)) == 1 or (max_depth is not None and depth == max_depth):
        return {"leaf": True, "pred": _majority(y), "n": len(y)}
    split = best_split(X, y, min_samples_leaf)
    if split is None:
        return {"leaf": True, "pred": _majority(y), "n": len(y)}
    j, t, g = split
    left = X[:, j] <= t
    return {
        "leaf": False, "feature": j, "threshold": t, "n": len(y),
        "left": build_tree(X[left], y[left], max_depth, min_samples_leaf, depth + 1),
        "right": build_tree(X[~left], y[~left], max_depth, min_samples_leaf, depth + 1),
    }


def _predict_one(tree, x):
    while not tree["leaf"]:
        tree = tree["left"] if x[tree["feature"]] <= tree["threshold"] else tree["right"]
    return tree["pred"]


def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    X = np.asarray(X)
    return np.array([_predict_one(tree, x) for x in X])


def count_leaves(tree):
    return 1 if tree["leaf"] else count_leaves(tree["left"]) + count_leaves(tree["right"])

In [6]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

toy = pd.DataFrame({
    "age_55_plus": [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "exercise_angina": [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "male": [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "high_bp": [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    "disease": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

# Q1: reproduce Part 1 values on the toy table
y_toy = toy["disease"].to_numpy()
print("entropy(toy):", entropy(y_toy))
for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    print(f"{f:16s} gain = {information_gain(y_toy, toy[f].to_numpy() == 0):.4f}")

# Q2: best_split vs sklearn depth-1 on the training split
mine = best_split(X_train, y_train)
ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED).fit(X_train, y_train)
print("mine:   ", feature_names[mine[0]], mine[1], round(mine[2], 4))
print("sklearn:", feature_names[ref.tree_.feature[0]], ref.tree_.threshold[0])

entropy(toy): 1.0
age_55_plus      gain = 0.3500
exercise_angina  gain = 0.4591
male             gain = 0.0000
high_bp          gain = 0.0817
mine:    ca 0.5 0.2509
sklearn: ca 0.5


In [7]:
import time

depths = [1, 2, 3, 4, 5, None]
rows = []
t0 = time.time()
for d in depths:
    mine_tree = build_tree(X_train, y_train, max_depth=d)
    ref_tree = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED).fit(X_train, y_train)
    p_mine = predict_tree(mine_tree, X_test)
    p_ref = ref_tree.predict(X_test)
    rows.append({
        "max_depth": "None" if d is None else d,
        "leaves_mine": count_leaves(mine_tree),
        "leaves_sklearn": ref_tree.get_n_leaves(),
        "test_agreement": float((p_mine == p_ref).mean()),
    })
agreement = pd.DataFrame(rows)
print(agreement.to_string(index=False))
print(f"time: {time.time() - t0:.1f}s")

max_depth  leaves_mine  leaves_sklearn  test_agreement
        1            2               2        1.000000
        2            4               4        1.000000
        3            8               8        1.000000
        4           15              15        1.000000
        5           21              21        0.977778
     None           32              32        0.966667
time: 0.2s


In [8]:
def compare_trees(my, sk, node, Xn, yn, path="root"):
    """Walk both trees together and report the first node where the chosen split differs."""
    sk_leaf = sk.children_left[node] == -1
    if my["leaf"] or sk_leaf:
        if my["leaf"] != sk_leaf:
            print(f"{path}: n={len(yn)}, one tree stops here and the other splits")
        return
    jm, tm = my["feature"], my["threshold"]
    js, ts = int(sk.feature[node]), float(sk.threshold[node])
    if jm != js or abs(tm - ts) > 1e-4 * max(1.0, abs(tm)):
        gm = information_gain(yn, Xn[:, jm] <= tm)
        gs = information_gain(yn, Xn[:, js] <= ts)
        print(f"{path}: n={len(yn)}")
        print(f"  mine:    {feature_names[jm]} <= {tm:.4f}, gain = {gm:.6f}")
        print(f"  sklearn: {feature_names[js]} <= {ts:.4f}, gain = {gs:.6f}")
        print(f"  gain difference = {abs(gm - gs):.2e}")
        return
    mask = Xn[:, jm] <= tm
    compare_trees(my["left"], sk, sk.children_left[node], Xn[mask], yn[mask], path + " -> L")
    compare_trees(my["right"], sk, sk.children_right[node], Xn[~mask], yn[~mask], path + " -> R")

for d in [5, None]:
    print(f"--- max_depth = {d} ---")
    mt = build_tree(X_train, y_train, max_depth=d)
    rt = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED).fit(X_train, y_train)
    compare_trees(mt, rt.tree_, 0, X_train, y_train)

--- max_depth = 5 ---
root -> L -> R -> L -> R: n=12
  mine:    cp <= 1.5000, gain = 0.247150
  sklearn: chol <= 275.5000, gain = 0.247150
  gain difference = 0.00e+00
root -> L -> R -> R -> R: n=5
  mine:    age <= 59.0000, gain = 0.321928
  sklearn: sex <= 0.5000, gain = 0.321928
  gain difference = 0.00e+00
root -> R -> L -> R -> L: n=4
  mine:    age <= 55.5000, gain = 0.311278
  sklearn: ca <= 1.5000, gain = 0.311278
  gain difference = 0.00e+00
--- max_depth = None ---
root -> L -> L -> L -> L -> L -> R: n=3
  mine:    trestbps <= 106.0000, gain = 0.918296
  sklearn: restecg <= 1.0000, gain = 0.918296
  gain difference = 0.00e+00
root -> L -> R -> L -> R: n=12
  mine:    cp <= 1.5000, gain = 0.247150
  sklearn: chol <= 275.5000, gain = 0.247150
  gain difference = 0.00e+00
root -> L -> R -> R -> R: n=5
  mine:    age <= 59.0000, gain = 0.321928
  sklearn: chol <= 214.0000, gain = 0.321928
  gain difference = 0.00e+00
root -> R -> L -> R -> L: n=4
  mine:    age <= 55.5000, gain =

**Agreement with scikit-learn.** For depths 1-4 my tree and scikit-learn's predict the same label on 100% of the test rows. At depth 5 the agreement is 97.8% (2 of 90 rows) and with unlimited depth 96.7% (3 of 90 rows). The number of leaves is identical in every case.

**Cause.** Walking both trees together shows that they first differ at small nodes (n = 3 to 12 training rows). At every such node the information gain of my split and of scikit-learn's split is exactly equal (difference 0.00e+00), i.e. it is a tie between several features that separate the few patients in the node in the same way. My implementation resolves ties by the rule from the task (lower feature index, then lower threshold). scikit-learn visits the features in a random order controlled by `random_state` and keeps the first best one, so on a tie it can pick another, equally good, feature. The trees are therefore equivalent in training quality, but the different split in one small node changes the prediction for the few test rows that fall into it.